<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/bioassay/bioassay_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bioassay: a compact Bayesian workflow

**Short Bayesian course — worked example**

Four groups of five animals were exposed to different doses and the number of deaths was recorded. We will use a binomial logistic model to ask:

1. How does mortality change with dose?
2. What dose gives a 50% mortality probability — the **LD50**?

The statistical workflow is the point of the example:

$$
\text{data}
\rightarrow
\text{model}
\rightarrow
\text{prior predictive}
\rightarrow
\text{fit and diagnose}
\rightarrow
\text{scientific quantity}
\rightarrow
\text{posterior predictive}.
$$

The data and model follow Gelman & Vehtari, *Bayesian Workflow*, §3.5.

## 0. Setup

This notebook uses the PyMC / modular ArviZ stack provided by Colab. Plotting and statistical summaries are delegated to arviz-plots and arviz-stats instead of being reconstructed with NumPy and Matplotlib.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import pymc as pm
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260923
azp.style.use("arviz-variat")

def add_interval_legend(ax, line_label="median", observed_label="observed"):
    """Compact legend for the 50%/90% intervals, median, and observations."""
    ax.legend(
        handles=[
            Line2D([0], [0], color="C1", lw=1.6, label=line_label),
            Patch(facecolor="C0", alpha=0.9, label="50% HDI"),
            Patch(facecolor="C0", alpha=0.28, label="90% HDI"),
            Line2D(
                [0], [0], marker="o", linestyle="none", color="black",
                markersize=4.5, label=observed_label,
            ),
        ],
        loc="upper left",
        ncols=2,
        fontsize=8,
        handlelength=1.4,
        handletextpad=0.45,
        columnspacing=0.9,
        borderpad=0.35,
        labelspacing=0.35,
        frameon=True,
        framealpha=0.88,
    )

print("PyMC:", pm.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## 1. Data

At dose $x_j$, we observe $y_j$ deaths among $n_j=5$ animals.

In [ ]:
dose = np.array([-0.86, -0.30, -0.05, 0.73])
n = np.array([5, 5, 5, 5])
deaths = np.array([0, 1, 3, 5])

bioassay = pd.DataFrame(
    {
        "dose_log_g_ml": dose,
        "animals": n,
        "deaths": deaths,
    }
)
bioassay["proportion_dead"] = bioassay["deaths"] / bioassay["animals"]
bioassay

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(dose, deaths / n, s=70)
ax.set(
    xlabel="Dose log(g/ml)",
    ylabel="Observed proportion dead",
    ylim=(-0.05, 1.05),
)
plt.show()

## 2. Generative model

For group $j$,

$$
y_j \sim \operatorname{Binomial}(n_j,p_j),
\qquad
\operatorname{logit}(p_j)=\alpha+\beta x_j.
$$

We use

$$
\alpha\sim N(0,5),
\qquad
\beta\sim \operatorname{HalfNormal}(5).
$$

The positive support of $\beta$ encodes the substantive assumption that mortality does not decrease as dose increases.

The mortality probability $p$ is retained because we use it to display the posterior dose-response relationship. LD50 is retained because it is the scientific quantity we will interpret later.

In [ ]:
coords = {"dose_log_g_ml": dose}

with pm.Model(coords=coords) as model:
    dose_data = pm.Data("dose", dose, dims="dose_log_g_ml")

    alpha = pm.Normal("alpha", mu=0, sigma=5)
    beta = pm.HalfNormal("beta", sigma=5)

    logit_p = alpha + beta * dose_data
    p = pm.Deterministic(
        "p",
        pm.math.sigmoid(logit_p),
        dims="dose_log_g_ml",
    )

    ld50_log_g_ml = pm.Deterministic(
        "LD50_log_g_ml",
        -alpha / beta,
    )
    ld50_mg_ml = pm.Deterministic(
        "LD50_mg_ml",
        1000 * pm.math.exp(ld50_log_g_ml),
    )

    pm.Binomial(
        "deaths",
        n=n,
        logit_p=logit_p,
        observed=deaths,
        dims="dose_log_g_ml",
    )

## 3. Prior predictive check

Before fitting, inspect the observable implications of the priors.

> **What death counts do these priors say are plausible?**

The intervals summarize replicated **death counts** $y$. Because $y$ is discrete, an HDI is the shortest interval containing the requested probability and need not be centered on the median. The median can therefore coincide with one endpoint of a 50% HDI. Vertical intervals are a better visual match to four discrete Binomial outcomes than a filled ribbon.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=1000,
        var_names=["deaths"],
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_lm(
    prior,
    x="dose",
    y="deaths",
    y_obs="deaths",
    group="prior_predictive",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    visuals={
        "ci_band": False,
        "ci_vlines": {"color": "C0"},
        "pe_line": {"color": "C1"},
        "observed_scatter": {"color": "black", "alpha": 1},
    },
)

ax = plt.gca()
ax.set(xlabel="Dose log(g/ml)", ylabel="Deaths out of 5", ylim=(-0.25, 5.25))
add_interval_legend(ax, line_label="median", observed_label="observed")
plt.show()

The darker vertical interval is the 50% prior predictive HDI and the lighter interval is the 90% HDI. The line joins the predictive medians. These intervals include Binomial outcome variability because they summarize replicated death counts, not only uncertainty about the mortality probability.

## 4. Fit and diagnose

Now condition on the observations. PyMC returns an xarray DataTree containing the posterior and sampler diagnostics.

Before interpreting the model, inspect divergences, $\hat R$, effective sample sizes, and the traces.

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        nuts={"target_accept": 0.90},
        random_seed=RANDOM_SEED,
    )

In [ ]:
print("Divergences:", idata["sample_stats"]["diverging"].sum().item())

azs.summary(
    idata,
    var_names=["alpha", "beta"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["alpha", "beta"],
)

## 5. Posterior dose-response fit

First look at the fitted relationship itself. For each dose, the model implies an expected number of deaths $5p$.

The intervals here describe **posterior uncertainty about the mean response**. They do not include the additional Binomial variation in an actual group of five animals.

In [ ]:
idata["posterior"]["expected_deaths"] = 5 * idata["posterior"]["p"]

azp.plot_lm(
    idata,
    x="dose",
    y="expected_deaths",
    y_obs="deaths",
    group="posterior",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    visuals={
        "pe_line": {"color": "C1"},
        "ci_band": {"color": "C0"},
        "observed_scatter": {"color": "black", "alpha": 1},
    },
)

ax = plt.gca()
ax.set(xlabel="Dose log(g/ml)", ylabel="Deaths out of 5", ylim=(-0.25, 5.25))
add_interval_legend(ax, line_label="median 5p", observed_label="observed")
plt.show()

## 6. Scientific quantity: LD50

The LD50 is the dose for which $p=0.5$. Because $\operatorname{logit}(0.5)=0$,

$$
0=\alpha+\beta\,LD50
\quad\Longrightarrow\quad
LD50=-\frac{\alpha}{\beta}.
$$

Because LD50 was declared inside the model, it already has a posterior distribution. We can hand that distribution directly to ArviZ.

In [ ]:
azs.summary(
    idata,
    var_names=["LD50_log_g_ml", "LD50_mg_ml"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_dist(
    idata,
    var_names=["LD50_mg_ml"],
    point_estimate="median",
    ci_prob=0.90,
    ci_kind="hdi",
)

## 7. Posterior predictive check

A posterior distribution for the mean response is not yet a check of the observation model. Generate new death counts at the **same four doses** from the fitted model.

This plot deliberately uses the same format as the prior predictive check. The difference from the posterior dose-response plot is substantive: these HDIs are for replicated $y$, so they include both posterior uncertainty about $p$ and Binomial variation among groups of five animals.

In [ ]:
with model:
    pm.sample_posterior_predictive(
        idata,
        var_names=["deaths"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

In [ ]:
azp.plot_lm(
    idata,
    x="dose",
    y="deaths",
    y_obs="deaths",
    group="posterior_predictive",
    plot_dim="dose_log_g_ml",
    ci_prob=(0.50, 0.90),
    ci_kind="hdi",
    point_estimate="median",
    smooth=False,
    visuals={
        "ci_band": False,
        "ci_vlines": {"color": "C0"},
        "pe_line": {"color": "C1"},
        "observed_scatter": {"color": "black", "alpha": 1},
    },
)

ax = plt.gca()
ax.set(xlabel="Dose log(g/ml)", ylabel="Deaths out of 5", ylim=(-0.25, 5.25))
add_interval_legend(ax, line_label="median", observed_label="observed")
plt.show()

The question is whether the observed counts look ordinary relative to datasets the fitted model can generate. With four groups this is necessarily a modest check, but it makes the distinction between uncertainty in the fitted mean and uncertainty in future observations explicit.

## What to carry forward

This example establishes a reusable workflow:

- specify a generative model;
- inspect implications of the priors on the observable scale;
- fit and diagnose before interpreting;
- distinguish posterior uncertainty about the fitted mean from posterior predictive variability;
- retain scientifically meaningful derived quantities in the model; and
- generate replicated data to check what the fitted model predicts.

### Optional explorations

1. Replace HalfNormal(5) for $\beta$ with Normal(0, 5). Does the data rule out a decreasing relationship?
2. Try narrower priors and rerun the **prior predictive check before fitting**.

## Sources

- Gelman, A. & Vehtari, A. *Bayesian Workflow*, §3.5, “Bioassay case study.”
- Racine-Poon, A., Grieve, A. P., Fluhler, H., & Smith, A. F. M. (1986). “Bayesian Methods in Practice: Experiences in the Pharmaceutical Industry.” *Applied Statistics*, 35, 93–150.
- PyMC 6 documentation.
- ArviZ plots/stats documentation.